# Annotation pipeline runner

Run the annotation workflow notebooks in order. The child notebooks execute in place, so their outputs remain visible in `notebooks/annotations/` after the pipeline completes.

> **Warning:** all listed stages run by default. This includes downloading data, local model inference, MP4/R2 upload, and loading the Bronze database tables. Set `STAGES_TO_RUN` to a subset for a partial rerun.

Every stage has its own `OVERWRITE` variable (default `False`: only process what is missing or out of date). `OVERWRITE` in this runner overrides all of them for this run, via the `PIPELINE_OVERWRITE` environment variable:

- `None`: each notebook uses its own setting.
- `False`: every stage only processes what is missing.
- `True`: every stage rebuilds everything (re-downloads, re-runs the Qwen activity gate, YOLO and Qwen annotation, drops and reloads the bronze tables).

Stages 04 and 05 both load the local MLX vision model: 04 scores every 10 s chunk of each video for activity and keeps the ones above the threshold, 05 annotates the kept windows. Both must run natively on Apple Silicon.

In [3]:
import os
import subprocess
from pathlib import Path
from time import perf_counter

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
ANNOTATION_NOTEBOOKS = PROJECT_ROOT / "notebooks" / "annotations"
PROJECT_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
KERNEL_NAME = "annotation-pipeline"
if not PROJECT_PYTHON.is_file():
    raise FileNotFoundError(f"Project Python environment not found: {PROJECT_PYTHON}")

STAGES = [
    #"01_ingest_annotations_and_metadata.ipynb",
    #"02_create_manifests.ipynb",
    "03_download_videos.ipynb",
    "04_prepare_video_windows.ipynb",
    #"05_annotate_prepared_windows.ipynb",
    #"06_export_annotation_bundles.ipynb",
    #"07_upload_exports_to_supabase.ipynb",
]
# Example partial rerun: STAGES_TO_RUN = STAGES[3:]
STAGES_TO_RUN = STAGES

# None: each notebook uses its own OVERWRITE. True/False: override OVERWRITE in every stage.
OVERWRITE = None

stage_env = os.environ.copy()
if OVERWRITE is None:
    stage_env.pop("PIPELINE_OVERWRITE", None)
else:
    stage_env["PIPELINE_OVERWRITE"] = "true" if OVERWRITE else "false"
print("=" * 72, flush=True)
print("Annotation pipeline runner", flush=True)
print(f"Pipeline OVERWRITE override: {OVERWRITE}", flush=True)
print(f"Selected stages ({len(STAGES_TO_RUN)}): {', '.join(STAGES_TO_RUN)}", flush=True)
print("=" * 72, flush=True)

stage_paths = [ANNOTATION_NOTEBOOKS / stage for stage in STAGES_TO_RUN]
missing = [str(path) for path in stage_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing pipeline notebook(s):\n" + "\n".join(missing))

print(f"Installing/updating the {KERNEL_NAME!r} kernel from {PROJECT_PYTHON}...", flush=True)
subprocess.run(
    [
        str(PROJECT_PYTHON), "-m", "ipykernel", "install", "--user",
        "--name", KERNEL_NAME, "--display-name", "Annotation Pipeline (.venv)",
    ],
    check=True,
)

started = perf_counter()
for index, notebook_path in enumerate(stage_paths, start=1):
    print(f"\n{'=' * 72}\nStage {index}/{len(stage_paths)}: {notebook_path.name}\n{'=' * 72}", flush=True)
    subprocess.run(
        [
            str(PROJECT_PYTHON),
            "-m",
            "jupyter",
            "nbconvert",
            "--to",
            "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=-1",
            f"--ExecutePreprocessor.kernel_name={KERNEL_NAME}",
            str(notebook_path),
        ],
        cwd=PROJECT_ROOT,
        env=stage_env,
        check=True,
    )
    print(f"Completed: {notebook_path.name}", flush=True)

print(f"\nPipeline completed in {perf_counter() - started:.1f} seconds.", flush=True)


Installing/updating the 'annotation-pipeline' kernel from /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/bin/python...
Installed kernelspec annotation-pipeline in /Users/juanvargas/Library/Jupyter/kernels/annotation-pipeline

Stage 1/2: 03_download_videos.ipynb


[NbConvertApp] Converting notebook /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/notebooks/annotations/03_download_videos.ipynb to notebook
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


Completed: 03_download_videos.ipynb

Stage 2/2: 04_prepare_video_windows.ipynb


[NbConvertApp] Writing 7208 bytes to /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/notebooks/annotations/03_download_videos.ipynb
[NbConvertApp] Converting notebook /Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/notebooks/annotations/04_prepare_video_windows.ipynb to notebook
[IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.


KeyboardInterrupt: 

[IPKernelApp] ERROR | Exception in message handler:
Traceback (most recent call last):
  File "/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py", line 478, in dispatch_shell
    await result
  File "/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/ipykernel/ipkernel.py", line 372, in execute_request
    await super().execute_request(stream, ident, parent)
  File "/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py", line 857, in execute_request
    reply_msg: dict[str, t.Any] = self.session.send(  # type:ignore[assignment]
                                  ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        stream,
        ^^^^^^^
    ...<4 lines>...
        ident=ident,
        ^^^^^^^^^^^^
    )
    ^
  File "/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.